# Week 6 - Linear Regression

Learning contents:

1. Linear models
    - Polynomial function
    - Radial basis function
    - Sigmoidal basis function
    - Optimization of Error function
    - Test models
2. Bayesian Linear Regression 
    - Generate data
    - Fit the data
    - Predictive distribution

## Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import exp
from scipy.stats import multivariate_normal

import seaborn as sns; sns.set_theme(); sns.set_palette('bright')

## Generate Data

In [ ]:
def target_func(x): return np.sin(2*np.pi*x)

def generate_data(size):
    rng = np.random.RandomState(26052605)
    x_train = rng.uniform(0., 1., size)
    y_train = target_func(x_train) + rng.normal(scale=0.1, size=size)
    x_test = np.linspace(0., 1., 100)
    y_test = target_func(x_test)
    
    return x_train, y_train, x_test, y_test

x_train, y_train, x_test, y_test = generate_data(10)

In [ ]:
plt.scatter(x_train, y_train);

In [ ]:
plt.plot(x_test, y_test, '-');

## 1) Linear models

Write down the function named linear that takes data point x (scalar), a basis function, and weights (list|ndarray, shape (M,) — M weights where index 0 is the bias) and returns the output of linear basis function model (2nd equation in slide 5 of lecture 11, eq (3.3) in text book). The basis function (Phi(x)) that goes as input to linear will be defined later but it takes x (scalar) and i (index) and returns Phi_i(x) (which is a scalar).

In [36]:
def linear(x, basis, weights):
    # Convention: index 0 is the constant bias phi_0(x) = 1;
    # the remaining weights multiply basis(x, i) for i = 1, ..., M-1.
    pass


### 1.1) Polynomial basis function

In `create_polynomial` below, write the code of `polynomial` function that takes data point `x` (scalar) and `i` (index) and returns the i-th value (scalar) of a polynomial basis function Phi_i(x) (see equation in slide 7 of lecture 11). As you can see, `create_polynomial` returns the resulting function (polynomial)! 

In [ ]:
def create_polynomial():
    def polynomial(x, i):
        pass
    
    return polynomial

In [ ]:
# Self-check: phi_i(x) = x**i, so phi_3(2) should be 8
_poly = create_polynomial()
assert abs(_poly(2.0, 3) - 8.0) < 1e-10, "polynomial(2.0, 3) should be 2**3 = 8"
print("OK: polynomial basis self-check passed")


Plot the different polynomial basis functions for input x ranging from -1 to 1.

In [ ]:
x = np.linspace(-1, 1, 100)
n = 11

polynomial = create_polynomial()

for i in range(n):
    y = list(map(lambda x: polynomial(x, i), x))
    plt.plot(x, y, '-')

### 1.2) Radial basis function

In the create_radial function below, you will implement the inner function radial. The inner function takes two inputs: a data point x (scalar) and an index i. It should return the scalar value of the i-th radial basis function, as defined in slide 8 of Lecture 11.

The outer create_radial function takes as input a vector means and a scalar variance.

The vector means defines the centers of all the radial basis functions.

By passing the index i, the inner radial function selects the i-th mean (means[i]) and uses it, together with the common variance, to compute the value of the radial basis function.

Thus, calling radial(x, i) evaluates the i-th basis function at the point x, and by looping over different indices i, you can generate all radial basis functions centered at the values in means.

In [ ]:
def create_radial(means, variance):
    def radial(x, i):
        pass
    
    return radial

The code below demonstrates how to generate and plot multiple radial basis functions.

np.linspace(-1, 1, 100) creates the data points, x, where the functions will be evaluated.

np.linspace(-1, 1, 11) defines 11 equally spaced centers (means) for the radial basis functions.

create_radial constructs the inner function radial(x, i), which evaluates the i-th radial basis function at point 𝑥, using the corresponding mean and a fixed variance.

The loop over i calls radial(x, i) for each center and plots all 11 radial basis functions on the same figure.

This way you can see how each basis function is shaped and how they overlap across the interval [−1,1].

In [ ]:
x = np.linspace(-1, 1, 100)
n = 11

radial = create_radial(np.linspace(-1, 1, 11), 0.1)

for i in range(n):
    y = list(map(lambda x: radial(x, i), x))
    plt.plot(x, y, '-')

### 1.3) Sigmoidal basis function

In the create_sigmoidal function below, you will implement the inner function sigmoidal.

The outer function takes as input a vector of means (means) and a slope parameter (s).

The inner function sigmoidal(x, i) takes a data point x (scalar) and an index i.

It then computes the i-th sigmoidal basis function at point x, using the i-th mean from the vector means and the slope parameter s (see definition in slide 9 of Lecture 11).

In [ ]:
def create_sigmoidal(means, s):
    def sigmoidal(x, i):
        pass
    
    return sigmoidal

Like previous case, by looping over different indices i, you can plot all sigmoidal basis functions centered at the values in means.

In [ ]:
x = np.linspace(-1, 1, 100)
n = 11

sigmoidal = create_sigmoidal(np.linspace(-1, 1, 11), 0.1)

for i in range(n):
    y = list(map(lambda x: sigmoidal(x, i), x))
    plt.plot(x, y, '-')

### 1.4) Optimization of Error function

 
Write the `optimal_weights` function that computes the linear weights for fitting the given data with a chosen basis function (see the last equation on slide 11 of Lecture 11).  

**Details & conventions:**  
- `basis` is a function of the form `basis(x, i)` that evaluates the i-th basis function at scalar `x`.  
- We reserve **i = 0** for the bias term (phi_0(x) = 1).  
- For **i = 1 … M–1**, phi_i(x) = basis(x, i).  
- Construct the design matrix X with shape (N, M):  
  - Column 0 contains all ones (bias).  
  - Columns 1 … M–1 contain the basis function values evaluated at the training inputs.  
- Use this design matrix together with the target values to compute and return the weight vector `w`.  
- The result is a vector of length **M** (one bias + M–1 basis weights).  

**Inputs:**  
- `inputs`: list/1D array of training inputs of length N.  
- `targets`: list/1D array of training targets of length N.  
- `M`: total number of weights (1 bias + M–1 basis functions).  


In [ ]:
def optimal_weights(basis, inputs, targets, M):
    # Convention: column 0 of the design matrix is the constant bias phi_0(x) = 1;
    # columns 1, ..., M-1 hold basis(x, i) for i = 1, ..., M-1.
    pass


In [ ]:
# Self-check: optimal_weights should match numpy's least-squares solver
_basis = create_polynomial()
_w = optimal_weights(_basis, x_train, y_train, 4)
_X = np.array([[1 if i == 0 else _basis(xq, i) for i in range(4)] for xq in x_train])
_w_ref = np.linalg.lstsq(_X, y_train, rcond=None)[0]
assert np.allclose(_w, _w_ref), "optimal_weights should match np.linalg.lstsq"
print("OK: optimal_weights self-check passed")


### 1.5) Test models

In [ ]:
def test(basis, M, label=''):
    plt.plot(x_test, list(map(lambda x: linear(x, basis, optimal_weights(basis, x_train, y_train, M)), x_test)), '-', label=label)
    plt.plot(x_train, y_train, 'og')
    plt.legend()

The following code will test the performance of the optimal model using different basis function on input data generated at the start of the notebook. Which model you believe performs the best on this data? Does the model size affect the model performance?

In [ ]:
test(create_radial(np.linspace(-1, 1, 10), 0.1), 7, 'radial, M=7')
test(create_radial(np.linspace(-1, 1, 10), 0.1), 4, 'radial, M=4')

In [ ]:
test(create_polynomial(), 7, 'polynomial, M=7')
test(create_polynomial(), 4, 'polynomial, M=4')

In [ ]:
test(create_sigmoidal(np.linspace(0, 1, 8), 1), 7, 'sigmoidal, M=7')
test(create_sigmoidal(np.linspace(0, 1, 8), 1), 4, 'sigmoidal, M=4')

#### Your answer:

*Record here which basis/model combination fits the data best, and how the number of weights M affects the fit.*

## 2) Bayesian Linear Regression 

### 2.1) Generate data

The following functions generate training and test data corresponding to a linear function. The data corresponds to a regression problem where individual input x and output y are both scalars.

In [ ]:
def line_func(x): return -0.5 + 0.3 * x

def generate_line_data(size):
    rng = np.random.RandomState(26052605)
    x_train = rng.uniform(0., 1., size)
    y_train = line_func(x_train) + rng.normal(scale=0.05, size=size)
    x_test = np.linspace(0., 1., 100)
    y_test = line_func(x_test)
    
    return x_train, y_train, x_test, y_test

xl_train, yl_train, xl_test, yl_test = generate_line_data(25)

In [ ]:
plt.scatter(xl_train, yl_train);

In [ ]:
plt.scatter(xl_test, yl_test);

### 2.2) Fit the data

**Note on the meaning of M:** in this part `M` is the polynomial degree, so the weight vector has **M+1** components (w_0, ..., w_M) and `PHI` has **M+1** columns — unlike Part 1, where `M` counted the total number of weights (bias included).

Implement `bayesian_regression_fit(PHI, t, alpha, beta)` to return the posterior
mean and covariance of the weight vector for Bayesian linear regression (see
slide 23 in Lecture 12).

Inputs:
- PHI: design matrix of shape (N, D), where D is the number of weights
  (including the bias term; in the demo below D = M+1).
- t: targets, 1D array of length N (or shape (N, 1)).
- alpha: scalar prior precision for weights (zero-mean prior).
- beta: scalar noise precision (1/sigma^2).

Output:
- mean: ndarray of shape (D,) or (D,1) — posterior mean of the weights.
- covariance: ndarray of shape (D, D) — posterior covariance of the weights.

Notes:
- Use a zero-mean Gaussian prior with precision alpha * I_D.
- The posterior precision is alpha*I_D + beta * PHI.T @ PHI.
- Solve for the posterior mean using the targets and PHI.

Reference: PRML §3.3, eqs. (3.49)–(3.51).


In [ ]:
def bayesian_regression_fit(PHI, t, alpha, beta):
    pass

In [ ]:
# Self-check: with no data, the posterior should equal the prior N(0, (1/alpha) * I)
_m, _S = bayesian_regression_fit(np.zeros((0, 2)), np.zeros(0), alpha=2.0, beta=100.0)
assert np.allclose(_m, np.zeros(2)), "posterior mean with no data should be the prior mean (zeros)"
assert np.allclose(_S, np.eye(2) / 2.0), "posterior covariance with no data should be (1/alpha) * I"
print("OK: bayesian_regression_fit self-check passed")


In [ ]:
alpha = 1
beta = 100

x = np.linspace(-1, 1, 100)
w0, w1 = np.meshgrid(
    np.linspace(-1, 1, 100),
    np.linspace(-1, 1, 100)
)
w = np.array([w0, w1]).transpose(1, 2, 0)

basis = create_polynomial()

M = 1

# For the polynomial basis, phi_0(x) = x**0 = 1, so column 0 of PHI is the bias term naturally.
PHI = np.array([[basis(x[q], i) for q in range(len(x))] for i in range(M + 1)]).T
PHI_train = np.array([[basis(xl_train[q], i) for q in range(len(xl_train))] for i in range(M + 1)]).T

np.random.seed(26)  # make the sampled weight functions reproducible across runs
for end in [0, 1, 3, 6, 20]:
    
    w_mean, w_cov = bayesian_regression_fit(PHI_train[:end], yl_train[:end], alpha, beta)
    
    w_sample = np.random.multivariate_normal(
        w_mean.reshape(-1), w_cov, size=6
    )
    y_sample = PHI @ w_sample.T
    
    
    plt.subplot(1, 2, 1)
    plt.scatter(-0.5, 0.3, s=200, marker="x")
    plt.contour(w0, w1, multivariate_normal.pdf(w, mean=w_mean, cov=w_cov))
    plt.gca().set_aspect('equal')
    plt.xlabel("$w_0$")
    plt.ylabel("$w_1$")
    plt.title("prior/posterior")

    plt.subplot(1, 2, 2)
    plt.scatter(xl_train[:end], yl_train[:end], s=100, facecolor="none", edgecolor="steelblue", lw=1)
    plt.plot(x, y_sample, c="orange")
    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.show()

### 2.3) Predictive distribution

Implement the sample(PHI_train, y_train, alpha, beta, PHI_test) function to compute the posterior predictive distribution for test points in Bayesian linear regression (see slide 24 in Lecture 12).

Inputs:

PHI_train: design matrix of shape (N, D) for the training set.

y_train: targets, array of length N (or shape (N, 1)).

alpha: scalar prior precision for the weights.

beta: scalar noise precision (1/σ²).

PHI_test: design matrix of shape (N_test, D) for the test set.

Outputs:

y: array of length N_test — predictive means at the test inputs.

y_std: array of length N_test — predictive standard deviations at the test inputs.

Notes:

First, call bayesian_regression_fit with the training data to obtain the posterior mean and covariance of the weights.

Compute the predictive mean as
y = PHI_test @ w_mean.

For each test row φ*, compute the predictive variance as
(1 / beta) + φ*^T @ w_cov @ φ*.

Return y_std as the square root of these predictive variances.

Reference: PRML §3.3, eqs. (3.58)–(3.59).


In [ ]:
def sample(PHI_train, y_train, alpha, beta, PHI_test):
    pass

In [ ]:
alpha = 1e-3
beta = 2
x_train, y_train, x_test, y_test = generate_data(25)

basis = create_radial(np.linspace(0, 1, 9), 0.1)

M = 8

PHI_train = np.array([[basis(x_train[q], i) for q in range(len(x_train))] for i in range(M + 1)]).T
PHI_test = np.array([[basis(x_test[q], i) for q in range(len(x_test))] for i in range(M + 1)]).T

for end in [1, 2, 4, 8, 25]:
    
    y, y_std = sample(PHI_train[:end], y_train[:end], alpha, beta, PHI_test)
    
    plt.scatter(x_train[:end], y_train[:end], s=100, facecolor="none", edgecolor="steelblue", lw=2)
    plt.plot(x_test, y_test,'b')
    plt.plot(x_test, y,'r')
    plt.fill_between(x_test, y - y_std, y + y_std, color="orange", alpha=0.5)
    plt.xlim(0, 1)
    plt.ylim(-2, 2)
    plt.show()